# 03 — Cleaning (Scrub)

Takes the raw scraped strings from `data/raw/books_raw.csv` and converts them into an analysis-ready dataset saved to `data/processed/books_clean.csv`.

**Principle**: nothing is dropped silently. Every removal, conversion or flag is counted and reported, so the transformation from raw to clean is fully auditable.

In [1]:
import sys
from pathlib import Path
import re

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/books_raw.csv")
print(raw.shape)
raw.dtypes

(1000, 13)


title                        str
price_raw                    str
availability_raw             str
star_rating_word             str
detail_url                   str
category                     str
description                  str
upc                          str
price_excl_tax_raw           str
price_incl_tax_raw           str
tax_raw                      str
availability_detail_raw      str
num_reviews_raw            int64
dtype: object

## Step 1 — survey the raw data

Before writing any cleaning logic, check which columns actually carry information. A column with a single repeated value can't contribute to analysis and should be identified now rather than discovered halfway through the exploration.

In [2]:
for col in raw.columns:
    n_unique = raw[col].nunique(dropna=False)
    n_missing = raw[col].isna().sum()
    print(f"{col:28} unique={n_unique:5}  missing={n_missing}")

title                        unique=  999  missing=0
price_raw                    unique=  903  missing=0
availability_raw             unique=    1  missing=0
star_rating_word             unique=    5  missing=0
detail_url                   unique= 1000  missing=0
category                     unique=   50  missing=0
description                  unique=  999  missing=2
upc                          unique= 1000  missing=0
price_excl_tax_raw           unique=  903  missing=0
price_incl_tax_raw           unique=  903  missing=0
tax_raw                      unique=    1  missing=0
availability_detail_raw      unique=   21  missing=0
num_reviews_raw              unique=    1  missing=0


### Findings from the survey

| Column | Unique | Verdict |
|---|---|---|
| `availability_raw` | 1 | Constant (`In stock`) — no information, drop |
| `tax_raw` | 1 | Constant (`£0.00`) — no information, drop |
| `num_reviews_raw` | 1 | Constant (`0`) — the site records no reviews at all, drop |
| `price_excl_tax_raw` | 903 | Identical to `price_raw` (tax is always zero) — redundant, drop |
| `price_incl_tax_raw` | 903 | Identical to `price_raw` — redundant, drop |
| `upc` | 1000 | Fully unique → the real primary key |
| `title` | 999 | One duplicated title, with different UPCs |
| `description` | 998 + 2 missing | Two books genuinely have no description on the site |
| `availability_detail_raw` | 21 | Real variation in stock counts — the usable availability signal |
| `category` | 50 | Key categorical variable for the analysis |

Five of thirteen columns carry zero information. Verifying this *before* the exploration
stage avoids producing meaningless charts of constant values later.

In [3]:
same_excl = (raw["price_raw"] == raw["price_excl_tax_raw"]).all()
same_incl = (raw["price_raw"] == raw["price_incl_tax_raw"]).all()
print("price_raw == price_excl_tax_raw:", same_excl)
print("price_raw == price_incl_tax_raw:", same_incl)
print("tax values:", raw["tax_raw"].unique())
print("availability values:", raw["availability_raw"].unique())
print("review values:", raw["num_reviews_raw"].unique())

price_raw == price_excl_tax_raw: True
price_raw == price_incl_tax_raw: True
tax values: <StringArray>
['£0.00']
Length: 1, dtype: str
availability values: <StringArray>
['In stock']
Length: 1, dtype: str
review values: [0]


In [4]:
print("Books with no description:")
print(raw.loc[raw["description"].isna(), ["title", "category", "detail_url"]])

dup_titles = raw[raw["title"].duplicated(keep=False)]
print("\nDuplicated title rows:")
print(dup_titles[["title", "upc", "price_raw", "category"]])

Books with no description:
                                                 title  category  \
160  The Bridge to Consciousness: I'm Writing the B...   Default   
995  Alice in Wonderland (Alice's Adventures in Won...  Classics   

                                            detail_url  
160  https://books.toscrape.com/catalogue/the-bridg...  
995  https://books.toscrape.com/catalogue/alice-in-...  

Duplicated title rows:
                      title               upc price_raw category
236  The Star-Touched Queen  1528279aec1f3dce    £46.02  Fantasy
358  The Star-Touched Queen  4a7a25be293ad678    £32.30  Fantasy


### Duplicate and missing-value decisions

**Duplicate title** — "The Star-Touched Queen" appears twice, but with different UPCs
(`1528279aec1f3dce` / `4a7a25be293ad678`) and different prices (£46.02 / £32.30).
These are two separate catalogue entries, not a scraping artefact. **Both rows are kept.**
`upc` is used as the primary key rather than `title`.

**Missing descriptions** — two books have no description on the site itself. Since
description is not used in the quantitative analysis, these rows are kept and the field
is left as `NaN` rather than imputed with a placeholder. **No rows are dropped.**

Final row count after cleaning: still 1000.

## Step 2 — regex cleaning

Three fields need parsing out of text:

| Field | Raw | Target |
|---|---|---|
| price | `£51.77` | `51.77` (float) |
| stock | `In stock (22 available)` | `22` (int) |
| rating | `Three` | `3` (int) |

Price and stock use regex; the rating is a fixed vocabulary of five words, so a lookup
dictionary is both clearer and safer than a pattern.

In [5]:
def parse_price(raw_price: str) -> float:
    """'£51.77' -> 51.77"""
    match = re.search(r"(\d+\.\d+)", raw_price)
    return float(match.group(1)) if match else np.nan


def parse_stock(raw_availability: str) -> int:
    """'In stock (22 available)' -> 22"""
    match = re.search(r"\((\d+)\s+available\)", raw_availability)
    return int(match.group(1)) if match else np.nan


RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_rating(word: str) -> int:
    """'Three' -> 3"""
    return RATING_MAP.get(word, np.nan)

In [6]:
print(parse_price("£51.77"), parse_price("£9.99"))
print(parse_stock("In stock (22 available)"), parse_stock("In stock (1 available)"))
print(parse_rating("Three"), parse_rating("Five"), parse_rating("Unknown"))

51.77 9.99
22 1
3 5 nan
